In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# 1. Define the path to your .env file
# pathlib handles spaces in folder names (e.g., 'analytic projects') seamlessly
env_path = (
    Path.home()
    / "Documents"
    / "analytics projects"
    / "tuberculosis_project"
    / ".env"
)

# 2. Load the environment variables
# override=True ensures existing environment variables are updated if needed
loaded = load_dotenv(dotenv_path=env_path, override=True)

# 3. Verify loading status
if loaded:
    print(f"✅ Successfully loaded CONFIG from: {env_path}")
else:
    print(f"❌ Failed to find or load .env at: {env_path}")

✅ Successfully loaded CONFIG from: C:\Users\Administrator\Documents\analytics projects\tuberculosis_project\.env


In [2]:
%%writefile config.py

import os
from dotenv import load_dotenv
from pathlib import Path

# Exact path matching your folder structure
env_path = Path(r"C:\Users\Administrator\Documents\analytics projects\tuberculosis_project\.env")
load_dotenv(dotenv_path=env_path)

DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST     = os.getenv("DB_HOST", "localhost")
DB_PORT     = os.getenv("DB_PORT", "5432")
DB_NAME     = os.getenv("DB_NAME")

missing = [k for k, v in {
    "DB_USER"    : DB_USER,
    "DB_PASSWORD": DB_PASSWORD,
    "DB_NAME"    : DB_NAME
}.items() if v is None]

if missing:
    raise ValueError(
        f"These variables were not loaded from .env: {missing}\n"
        f"Check that your .env file exists at: {env_path}"
    )

DB_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"


# config.py

CONFIG = {
    # --- Database ---
    "db_url"      : DB_URL,
    "raw_table"   : "tb_raw_data",
    "clean_table" : "tb_data_clean",
    "schema"      : "public",

    # --- Keys ---
    "primary_key" : None,        # no single primary key
                                 # composite: country+year+measure+age+sex+risk
    "country_key" : "iso3",
    "year_col"    : "year",

    # --- Target ---
    "estimate_col"   : "best",
    "lower_bound_col": "lo",
    "upper_bound_col": "hi",

    # --- Column groups ---
    "id_cols": ["iso2", "iso3", "iso_numeric"],

    "categorical_cols": [
        "country", "measure", "unit",
        "age_group", "sex", "risk_factor"
    ],

    "numeric_cols": ["best", "lo", "hi"],

    "dimension_cols": [
        "country", "iso3", "year",
        "measure", "age_group", "sex", "risk_factor"
    ],

    # --- Valid controlled vocabularies ---
    "valid_sex": ["Male", "Female", "Both sexes"],

    "valid_measures": [
        "Incidence", "Mortality", "Prevalence",
        "Case fatality ratio", "Treatment coverage"
    ],

    "valid_units": [
        "absolute number",
        "rate per 100 000 population",
        "proportion"
    ],

    # --- Reference values ---
    "baseline_year"       : 2015,    # WHO End TB baseline
    "analysis_start_year" : 2000,

    # WHO high-burden 30 countries (iso3)
    "who_high_burden_30": [
        "AGO", "BGD", "BRA", "CAF", "CHN", "COD", "PRK",
        "ETH", "IND", "IDN", "KEN", "LSO", "LBR", "MNG",
        "MOZ", "MMR", "NAM", "NGA", "PAK", "PNG", "PHL",
        "RUS", "SLE", "ZAF", "THA", "UGA", "TZA", "VNM",
        "ZMB", "ZWE"
    ],

    # East Africa focus countries
    "east_africa": [
        "KEN", "UGA", "TZA", "ETH", "RWA",
        "BDI", "SOM", "SSD", "ERI", "DJI"
    ],

    # WHO regions mapping
    "who_regions": {
        "AFRO" : ["KEN","NGA","ZAF","ETH","UGA","TZA",
                  "MOZ","ZMB","ZWE","AGO","COD","CAF",
                  "SLE","LSO","NAM","LBR","BWA","GHA"],
        "SEARO": ["IND","IDN","BGD","MMR","PRK","THA",
                  "NPL","LKA","BTN","TLS","MDV"],
        "WPRO" : ["CHN","PHL","VNM","PNG","MNG","KHM",
                  "KOR","JPN","AUS","NZL","MYS"],
        "AMRO" : ["BRA","PER","MEX","COL","BOL","HTI",
                  "ARG","USA","CAN","CUB"],
        "EURO" : ["RUS","UKR","GBR","DEU","FRA","ROM",
                  "KAZ","UZB","BLR","AZE"],
        "EMRO" : ["PAK","AFG","MAR","IRN","IRQ","YEM",
                  "SOM","SDN","EGY","TUN"]
    },

    # World Bank income groups
    "income_groups": {
        "Low Income"          : ["ETH","MOZ","COD","MWI",
                                 "UGA","TZA","RWA","MLI"],
        "Lower-Middle Income" : ["KEN","NGA","IND","BGD",
                                 "PAK","PHL","VNM","MMR"],
        "Upper-Middle Income" : ["ZAF","BRA","CHN","PER",
                                 "COL","THA","MNG","NAM"],
        "High Income"         : ["USA","GBR","DEU","FRA",
                                 "AUS","JPN","KOR","CAN"]
    },

    # End TB Strategy milestone targets
    # Percentage reduction from 2015 baseline
    "end_tb_targets": {
        2020: {"incidence": 20, "mortality": 35},
        2025: {"incidence": 50, "mortality": 75},
        2030: {"incidence": 80, "mortality": 90}
    }
}

Overwriting config.py


In [3]:
%%writefile data_dictionary.py

import pandas as pd
import os


# data_dictionary.py

DATA_DICTIONARY = {
    "country": {
        "type"       : "Categorical",
        "description": "Full country name as used by WHO",
        "notes"      : "Use iso3 for joining and grouping "
                       "— country names have inconsistent "
                       "formatting across WHO datasets"
    },
    "iso2": {
        "type"       : "Identifier",
        "description": "2-letter ISO country code",
        "notes"      : "Standardized — use for Power BI "
                       "map visual country matching"
    },
    "iso3": {
        "type"       : "Identifier",
        "description": "3-letter ISO country code",
        "notes"      : "Primary country identifier for "
                       "joining, grouping, and mapping. "
                       "KEN = Kenya"
    },
    "iso_numeric": {
        "type"       : "Identifier",
        "description": "Numeric ISO country code",
        "notes"      : "Cast to string — it is a lookup "
                       "code not a quantity"
    },
    "year": {
        "type"       : "Numeric — temporal",
        "description": "Year of TB burden estimate",
        "notes"      : "WHO End TB baseline year is 2015. "
                       "Analysis typically covers 2000 to "
                       "most recent available year"
    },
    "measure": {
        "type"       : "Categorical",
        "description": "The TB metric being estimated",
        "values"     : "Incidence, Mortality, Prevalence, "
                       "Case fatality ratio, "
                       "Treatment coverage",
        "notes"      : "Always filter by measure before "
                       "aggregating — mixing incidence and "
                       "mortality in one aggregation is a "
                       "critical analytical error"
    },
    "unit": {
        "type"       : "Categorical",
        "description": "Unit of the best, lo, hi estimates",
        "values"     : "absolute number, "
                       "rate per 100 000 population, "
                       "proportion",
        "notes"      : "Always filter by unit alongside "
                       "measure. Rate per 100,000 is used "
                       "for cross-country comparison. "
                       "Absolute numbers are used for "
                       "total burden assessment"
    },
    "age_group": {
        "type"       : "Ordinal categorical",
        "description": "Age band of the population estimate",
        "values"     : "0-4, 5-14, 15-24, 25-34, 35-44, "
                       "45-54, 55-64, 65+, all ages",
        "notes"      : "all ages aggregates across all bands. "
                       "Use specific bands for age-disaggregated "
                       "analysis. Never sum age bands that "
                       "include all ages — double counting"
    },
    "sex": {
        "type"       : "Categorical",
        "description": "Sex of the population estimate",
        "values"     : "Male, Female, Both sexes",
        "notes"      : "Both sexes aggregates male and female. "
                       "Never sum Male + Female + Both sexes "
                       "in the same query — triple counting"
    },
    "risk_factor": {
        "type"       : "Categorical",
        "description": "TB risk factor the estimate applies to",
        "values"     : "hiv, diabetes, alcohol, smoking, "
                       "undernutrition, all risk factors",
        "notes"      : "NULL or all risk factors = total burden "
                       "not attributable to a specific factor. "
                       "hiv = TB cases attributable to HIV "
                       "co-infection only. Never sum across "
                       "risk factors — overlapping populations"
    },
    "best": {
        "type"       : "Numeric",
        "description": "WHO point estimate (best estimate) "
                       "of the TB burden measure",
        "notes"      : "This is a modelled estimate not an "
                       "exact count. Must always be reported "
                       "alongside lo and hi confidence bounds. "
                       "Treating best as exact is a "
                       "methodological error"
    },
    "lo": {
        "type"       : "Numeric",
        "description": "Lower bound of 95% uncertainty interval",
        "notes"      : "lo > best indicates a data error — "
                       "flag these rows"
    },
    "hi": {
        "type"       : "Numeric",
        "description": "Upper bound of 95% uncertainty interval",
        "notes"      : "hi < best indicates a data error — "
                       "flag these rows. Wide hi-lo interval "
                       "indicates poor surveillance data quality"
    }
}

def save_data_dictionary(dictionary, filepath="outputs/data_dictionary.csv"):
    os.makedirs("outputs", exist_ok=True)
    rows = []
    for col, meta in dictionary.items():
        row = {"column": col}
        row.update(meta)
        rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(filepath, index=False)
    print(f"Data dictionary saved to {filepath}")
    print(f"Columns documented: {len(rows)}")
    return df

if __name__ == "__main__":
    dd_df = save_data_dictionary(DATA_DICTIONARY)
    print(dd_df[["column", "type", "description"]].to_string(index=False))

Overwriting data_dictionary.py
